# Classic SVM with Fisher Vectors on dataset `01_original`

**Why.** The improved SVM (`svm_improved`, handcrafted_v2, test macro-F1 0.581) still describes each image with global histograms
(whole image or a 2×2 grid). The hard classes (baby back ribs / prime rib / chicken wings, nachos / paella / guacamole) differ in *local* parts:
a bone end, a glaze, a cut surface. A global histogram averages those away.

**What.** The standard pre-deep-learning recipe for this (Sánchez et al., IJCV 2013; also the Food-101 paper's IFV baseline):

1. **Dense local descriptors.** On a 256 px padded image, a grid every 8 px at 3 patch sizes (16, 24, 32 px):
   - **RootSIFT** (SIFT, L1-normalised, square-rooted): local gradient shape;
   - **local LAB colour**: mean L, a\*, b\* in a 4×4 grid of cells over the same patch (48 numbers).
2. **PCA** (SIFT → 64, colour → 32), then append the patch's (x, y) position, so the encoding keeps rough layout.
3. **GMM vocabulary** (K Gaussians, diagonal covariance) fit on a sample of *training* descriptors only.
4. **Fisher Vector**: for each Gaussian, the average deviation of the image's descriptors from its mean and variance (2·K·D numbers),
   then power-normalisation (sign·√) and L2-normalisation.
5. **Linear SVM** on the Fisher Vectors (the usual pairing; the FV already plays the role of a kernel map).

Side check (cheap): **wavelet-energy texture** (3-level `db2` on L, a\*, b\*), added to handcrafted_v2.

Rules as before: dataset 01 only; every choice on **val**; test scored once per config; the chosen model (`svm_v3`) is compared
with `svm_improved` and the baseline `svm` by a paired test on the same 1,920 test images.
Outputs go to `results/01_original_improved/` under new names only (`svm3_*`, `svm2_v2*wav*`, `svm_v3`, `fv_*`).

In [ ]:
# ==== the only cell you normally edit ====
DATASET = "01_original"   # "01_original" | "02_denoise_clahe" | "03_denoise_clahe_gsam" | "04_denoise_clahe_dino"
RUN_MODE = "full"                  # "smoke": 20 imgs/class/split, 2 epochs, to test the pipeline | "full": the real run
SEED = 42
RETRAIN = False                     # True: ignore saved checkpoints / grid results and train again

# ---- from here on: identical for every dataset (same as the 03 run) ----
IMG_SIZE = 224
PAD_FILL = (124, 116, 104)          # same grey as the GSAM background (used to pad every dataset to a square)
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LABEL_SMOOTHING = 0.1

CLASSICAL_FEATURES = "handcrafted"  # "handcrafted" (HSV hist + HOG + LBP) | "resnet50" (frozen ImageNet embeddings)
LOGREG_C = [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100]
SVM_C = [0.1, 1, 10, 100, 1000]
SVM_GAMMA_MULT = [0.3, 1, 3]        # x 1/n_features (= sklearn "scale" after standardising)

CNN_GRID = [dict(n_blocks=n, base=32, dropout=0.3) for n in (3, 4, 5)]
CNN_TRAIN = dict(max_epochs=40, patience=6, lr=1e-3, weight_decay=5e-4)

RESNET_GRID = [dict(unfreeze="fc", lr=1e-3), dict(unfreeze="layer4", lr=3e-4), dict(unfreeze="all", lr=1e-4)]
RESNET_TRAIN = dict(max_epochs=15, patience=4, weight_decay=1e-4)

CLIP_ID = "openai/clip-vit-base-patch32"
TEMPLATE_SETS = {
    "classname": ["{}"],
    "a photo of": ["a photo of {}."],
    "food photo": ["a photo of {}, a type of food."],
    "ensemble": ["a photo of {}, a type of food.", "a close-up photo of {}.", "a photo of a plate of {}.",
                 "a good photo of {}.", "a cropped photo of {}."],
}

EXPECTED_MANIFEST_SHA256 = "79eb1d1cf8a475b8deaa2a301acc64a1c3a334fd4a289c8d45af91c359c3c2a4"
SMOKE_PER_CLASS = 20
if RUN_MODE == "smoke":
    CNN_TRAIN = {**CNN_TRAIN, "max_epochs": 2}
    RESNET_TRAIN = {**RESNET_TRAIN, "max_epochs": 2}
    LOGREG_C, SVM_C, SVM_GAMMA_MULT = [1e-2, 1], [1, 10], [1]

In [ ]:
# ==== paths (local VS Code or Colab) ====
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/food_classify")   # upload <DATASET>.zip here
    DATA_ROOT = Path("/content/data/dataset")
    if not (DATA_ROOT / DATASET).exists():
        import zipfile
        with zipfile.ZipFile(DRIVE_DIR / f"{DATASET}.zip") as z:
            z.extractall(DATA_ROOT)
    OUT_ROOT = DRIVE_DIR / "results"                               # on Drive, so it survives a disconnect
else:
    here = Path.cwd().resolve()
    PROJECT = next((p for p in [here, *here.parents] if (p / "data" / "dataset" / DATASET).exists()), None)
    assert PROJECT is not None, f"data/dataset/{DATASET} not found above {here}: unzip the dataset into data/dataset/"
    DATA_ROOT = PROJECT / "data" / "dataset"
    OUT_ROOT = PROJECT / "results"
    os.environ.setdefault("TORCH_HOME", str(PROJECT / "checkpoints" / "torch"))  # reuse downloaded weights
    os.environ.setdefault("HF_HOME", str(PROJECT / "checkpoints" / "hf"))

META = DATA_ROOT / "meta"
OUT = (OUT_ROOT / DATASET) if RUN_MODE == "full" else (OUT_ROOT / "smoke" / DATASET)
CKPT_DIR, FIG_DIR = OUT / "checkpoints", OUT / "figures"
for d in (OUT, CKPT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("data   :", DATA_ROOT / DATASET)
print("results:", OUT)

In [ ]:
# outputs go to the improvement folder (new names only), never into the baseline folder
BASE_OUT = OUT                                   # baseline results of the team notebook (read only)
OUT = OUT.parent / f"{DATASET}_improved"
FIG_DIR = OUT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
assert (BASE_OUT / "summary_test.csv").exists(), f"run the baseline first: {BASE_OUT}"
assert (OUT / "svm_improved" / "predictions_test.csv").exists(), f"run improve_01_original.ipynb first: {OUT}"
print("baseline:", BASE_OUT)
print("improved:", OUT)

# settings of the copied SVM cells
SVM2_C, SVM2_GAMMA_MULT = [1, 10, 100, 1000], [0.3, 1, 3]
# Fisher Vector settings
FV_SIZE, FV_STEP, FV_SCALES = 256, 8, (16, 24, 32)    # image side, grid step, patch sizes (px)
FV_PCA = {"sift": 64, "color": 32}
FV_K = [64, 128]                                       # GMM sizes tried
FV_SAMPLE_IMGS, FV_SAMPLE_PER_IMG = 2000, 120          # training descriptors used to fit PCA + GMM
FV_C = [0.01, 0.1, 1, 10]                              # LinearSVC C
if RUN_MODE == "smoke":
    SVM2_C, SVM2_GAMMA_MULT = [10], [1]
    FV_K, FV_SAMPLE_IMGS, FV_SAMPLE_PER_IMG, FV_C = [8], 50, 60, [1]

In [ ]:
import json, time, random, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support, f1_score, average_precision_score, confusion_matrix
from sklearn.preprocessing import label_binarize

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"                    # fp16 helps on CUDA; it did not help on the M1
NON_BLOCKING = DEVICE.type == "cuda"
NUM_WORKERS = 0 if sys.platform == "darwin" else 2  # macOS Jupyter can't pickle classes defined in a notebook into workers
pd.set_option("display.precision", 4)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=120, bbox_inches="tight")


def finish(fig, title, name):
    """Lay out a grid figure with its title above the first row (no overlap), save, show."""
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.55 / fig.get_figheight()))
    fig.suptitle(title, y=1 - 0.12 / fig.get_figheight())
    savefig(fig, name)
    plt.show()


seed_everything()
print(f"python {sys.version.split()[0]} ({sys.executable}) | torch {torch.__version__} | torchvision {torchvision.__version__} | device {DEVICE}")

In [ ]:
sha = hashlib.sha256((META / "manifest.csv").read_bytes()).hexdigest()
assert sha == EXPECTED_MANIFEST_SHA256, f"manifest differs from the team's ({sha}): wrong zip?"

df_all = pd.read_csv(META / "manifest.csv")
df_all["path"] = [str(DATA_ROOT / DATASET / r) for r in df_all.relpath]
missing = [p for p in df_all.path if not os.path.exists(p)]
assert not missing, f"{len(missing)} images in the manifest are missing, e.g. {missing[:3]}"
on_disk = {str(p.relative_to(DATA_ROOT / DATASET)) for p in (DATA_ROOT / DATASET).glob("*/*/*.jpg")}
assert on_disk == set(df_all.relpath), "folder contents and manifest differ"
assert (df_all.relpath.str.split("/").str[0] == df_all.split).all() and \
       (df_all.relpath.str.split("/").str[1] == df_all.label).all(), "folder and manifest split/label disagree"

CLASSES = sorted(df_all.label.unique())
K = len(CLASSES)
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
df_all["y"] = df_all.label.map(CLS2IDX)

key = df_all.label + "/" + df_all.relpath.str.split("/").str[-1]
assert not key.duplicated().any(), "the same image id appears in more than one split"

counts = df_all.pivot_table(index="label", columns="split", values="relpath", aggfunc="count")[["train", "val", "test"]]
counts.loc["total"] = counts.sum()
display(counts)
assert counts.drop("total").nunique().eq(1).all(), "classes are not balanced"
print("split ratio:", (counts.loc["total"] / counts.loc["total"].sum()).round(3).to_dict())

df = df_all.groupby(["split", "label"], group_keys=False).head(SMOKE_PER_CLASS) if RUN_MODE == "smoke" else df_all
splits = {s: df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test")}
y_split = {s: v.y.values for s, v in splits.items()}
print({s: len(v) for s, v in splits.items()}, "images used in this run")

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
CLIP_MEAN, CLIP_STD = [0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711]


def pad_to_square(img, fill=PAD_FILL):
    w, h = img.size
    s = max(w, h)
    out = Image.new("RGB", (s, s), fill)
    out.paste(img, ((s - w) // 2, (s - h) // 2))
    return out


train_tf = T.Compose([T.Lambda(pad_to_square),
                      T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
                      T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf = T.Compose([T.Lambda(pad_to_square), T.Resize((IMG_SIZE, IMG_SIZE)),
                     T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
clip_tf = T.Compose([T.Lambda(pad_to_square), T.Resize((IMG_SIZE, IMG_SIZE)),
                     T.ToTensor(), T.Normalize(CLIP_MEAN, CLIP_STD)])


def show_tensor(t, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Normalised CHW tensor -> HWC image in [0, 1] for plotting."""
    x = t.cpu() * torch.tensor(std)[:, None, None] + torch.tensor(mean)[:, None, None]
    return x.clamp(0, 1).permute(1, 2, 0).numpy()


class FoodDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths, self.labels, self.transform = frame.path.tolist(), frame.y.tolist(), transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(Image.open(self.paths[i]).convert("RGB")), self.labels[i]


def make_loader(split, transform, shuffle=False, batch_size=None):
    return DataLoader(FoodDataset(splits[split], transform), batch_size=batch_size or (BATCH_SIZE if shuffle else EVAL_BATCH_SIZE),
                      shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                      persistent_workers=NUM_WORKERS > 0)


# what the models see
demo = splits["train"].sample(3, random_state=SEED)
fig, axes = plt.subplots(3, 5, figsize=(12, 7.5))
for row, (_, r) in zip(axes, demo.iterrows()):
    img = Image.open(r.path).convert("RGB")
    row[0].imshow(img); row[0].set_title(f"stored {img.size}", fontsize=8)
    row[1].imshow(show_tensor(eval_tf(img))); row[1].set_title("eval 224x224", fontsize=8)
    for ax in row[2:]:
        ax.imshow(show_tensor(train_tf(img))); ax.set_title("train aug", fontsize=8)
    for ax in row: ax.axis("off")
plt.tight_layout(); savefig(fig, "transforms"); plt.show()

In [ ]:
RESULTS = {}   # model name -> {"val": {...}, "test": {...}, "config": {...}}


def topk_accuracy(y, scores, k):
    top = np.argsort(-scores, axis=1)[:, :k]
    return float((top == y[:, None]).any(axis=1).mean())


def compute_metrics(y, scores):
    y = np.asarray(y)
    pred = scores.argmax(1)
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    Y = label_binarize(y, classes=np.arange(K))
    ap = np.array([average_precision_score(Y[:, c], scores[:, c]) for c in range(K)])
    pc_p, pc_r, pc_f, support = precision_recall_fscore_support(y, pred, labels=np.arange(K), zero_division=0)
    per_class = pd.DataFrame({"precision": pc_p, "recall": pc_r, "f1": pc_f, "AP": ap, "support": support}, index=CLASSES)
    summary = {"top1": topk_accuracy(y, scores, 1), "top3": topk_accuracy(y, scores, 3), "top5": topk_accuracy(y, scores, 5),
               "macro_precision": float(p), "macro_recall": float(r), "macro_f1": float(f1), "mAP": float(np.nanmean(ap)), "n": int(len(y))}
    return summary, per_class


def macro_f1(y, scores):
    return f1_score(y, scores.argmax(1), average="macro", zero_division=0)


def plot_confusion(y, pred, title, name):
    cm = confusion_matrix(y, pred, labels=np.arange(K))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
    for ax, m, fmt, sub in [(axes[0], cm, "d", "counts"), (axes[1], cmn, ".2f", "row-normalised (diagonal = recall)")]:
        im = ax.imshow(m, cmap="Blues")
        ax.set_xticks(range(K), CLASSES, rotation=45, ha="right"); ax.set_yticks(range(K), CLASSES); ax.grid(False)
        for i in range(K):
            for j in range(K):
                ax.text(j, i, format(m[i, j], fmt), ha="center", va="center", fontsize=8,
                        color="white" if m[i, j] > m.max() / 2 else "black")
        ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{title}: {sub}")
        fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); savefig(fig, f"{name}_confusion"); plt.show()
    off = [(CLASSES[i], CLASSES[j], int(cm[i, j])) for i in range(K) for j in range(K) if i != j and cm[i, j] > 0]
    return pd.DataFrame(sorted(off, key=lambda t: -t[2])[:8], columns=["true", "predicted", "count"])


def show_predictions(frame, y, scores, correct, title, name, n=12, is_prob=True):
    pred = scores.argmax(1)
    idx = np.flatnonzero((pred == y) if correct else (pred != y))
    if len(idx) == 0:
        print("no", "correct" if correct else "wrong", "predictions"); return
    idx = np.random.default_rng(SEED).choice(idx, min(n, len(idx)), replace=False)
    cols = 6
    rows = int(np.ceil(len(idx) / cols))
    row_h = 3.3 if correct else 3.7          # room for the 2- or 3-line titles above each image
    fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, row_h * rows), squeeze=False)
    tag = "p" if is_prob else "s"
    for ax, i in zip(axes.ravel(), idx):
        top = np.argsort(-scores[i])[:2]
        ax.imshow(Image.open(frame.path[i]))
        txt = f"T: {CLASSES[y[i]]}\nP: {CLASSES[top[0]]} ({tag}={scores[i, top[0]]:.2f})"
        if not correct:
            txt += f"\n2nd: {CLASSES[top[1]]} ({scores[i, top[1]]:.2f})"
        ax.set_title(txt, fontsize=8, color="green" if correct else "red")
    for ax in axes.ravel(): ax.axis("off")
    finish(fig, title, name)


def save_predictions(name, split, y, scores):
    out = splits[split][["relpath", "label"]].copy()
    out["y_true"], out["y_pred"] = y, scores.argmax(1)
    out["pred_label"] = [CLASSES[i] for i in out.y_pred]
    for c in range(K):
        out[f"score_{CLASSES[c]}"] = scores[:, c]
    out.to_csv(OUT / name / f"predictions_{split}.csv", index=False)


def report_model(name, y_val, s_val, y_test, s_test, config, is_prob=True):
    """Final evaluation of a chosen model: val + test metrics, confusion matrix, examples, saved files."""
    (OUT / name).mkdir(exist_ok=True)
    sv, _ = compute_metrics(y_val, s_val)
    st, per_class = compute_metrics(y_test, s_test)
    RESULTS[name] = {"val": sv, "test": st, "config": config}
    display(pd.DataFrame([sv, st], index=["val", "test"]))
    display(per_class)
    display(plot_confusion(y_test, s_test.argmax(1), f"{name} (test)", name))
    show_predictions(splits["test"], y_test, s_test, True, f"{name}: correct test predictions", f"{name}_correct", is_prob=is_prob)
    show_predictions(splits["test"], y_test, s_test, False, f"{name}: wrong test predictions", f"{name}_wrong", is_prob=is_prob)
    save_predictions(name, "val", y_val, s_val)
    save_predictions(name, "test", y_test, s_test)
    per_class.to_csv(OUT / name / "per_class_test.csv")
    with open(OUT / name / "metrics.json", "w") as f:
        json.dump({"dataset": DATASET, "model": name, "seed": SEED, "run_mode": RUN_MODE, **RESULTS[name]}, f, indent=2, default=str)


def save_final(name, kind, model, arch=None, metrics=None):
    """Save a model as results/<dataset>/<name>/model.{pt,joblib} + config.json: the chosen model (after report_model) or any tried config.
    kind: "cnn" | "resnet50" | "clip" (torch, state_dict + architecture) or "sklearn" (whole fitted pipeline)."""
    import joblib
    d = OUT / name
    d.mkdir(parents=True, exist_ok=True)
    arch = json.loads(json.dumps(arch or {}, default=str))   # plain Python types, so torch.load(weights_only=True) can read it
    metrics = metrics or RESULTS.get(name, {})
    if kind == "sklearn":
        file = "model.joblib"
        joblib.dump(model, d / file)
    elif kind == "clip":
        file = "model.pt"
        torch.save({"kind": kind, "arch": arch, "classes": CLASSES, "text_emb": model.text_emb.cpu()}, d / file)
    else:
        file = "model.pt"
        torch.save({"kind": kind, "arch": arch, "classes": CLASSES,
                    "state_dict": {k: v.cpu() for k, v in model.state_dict().items()}}, d / file)
    config = {"dataset": DATASET, "model": name, "kind": kind, "file": file, "arch": arch, "classes": CLASSES,
              "preprocessing": {"pad_to_square_fill": PAD_FILL, "img_size": IMG_SIZE,
                                "normalize": "clip" if kind == "clip" else "imagenet",
                                "classical_features": CLASSICAL_FEATURES if kind == "sklearn" else None},
              "seed": SEED, "run_mode": RUN_MODE, "torch": torch.__version__,
              "val_macro_f1": metrics.get("val", {}).get("macro_f1"),
              "test_macro_f1": metrics.get("test", {}).get("macro_f1"),
              "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    with open(d / "config.json", "w") as f:
        json.dump(config, f, indent=2, default=str)
    size = (d / file).stat().st_size / 1e6
    print(f"saved {d / file} ({size:.1f} MB) + config.json")


ALL_DIR = OUT / "all_models"   # every config tried: model + config.json + metrics.json + scores.npz
ALL_DIR.mkdir(exist_ok=True)
CONFIGS, CHOSEN = {}, {}       # config name -> its val/test metrics; model type -> the config chosen on val


def config_done(name, arch):
    """Metrics of a config already trained, scored and saved with this arch, else None (then it is trained / scored now)."""
    p = ALL_DIR / name / "metrics.json"
    if RETRAIN or not p.exists() or not (ALL_DIR / name / "scores.npz").exists():
        return None
    res = json.loads(p.read_text())
    if res.get("arch") != json.loads(json.dumps(arch, default=str)):
        return None
    CONFIGS[name] = res
    return res


def record_config(name, family, kind, model, arch, scores, train_f1):
    """Score one tried config on val and test, save it to all_models/<name>/ and remember it.
    scores: {"val": (y, s), "test": (y, s)}. Test is recorded for the report; choosing uses val only."""
    arch = json.loads(json.dumps(arch, default=str))
    res = {"model": name, "family": family, "kind": kind, "arch": arch, "train_f1": float(train_f1),
           **{sp: compute_metrics(*scores[sp])[0] for sp in ("val", "test")}}
    save_final(f"all_models/{name}", kind, model, arch, metrics=res)
    np.savez(ALL_DIR / name / "scores.npz", **{f"{sp}_{k}": v for sp, (y, s) in scores.items() for k, v in (("y", y), ("s", s))})
    with open(ALL_DIR / name / "metrics.json", "w") as f:
        json.dump({"dataset": DATASET, **res}, f, indent=2, default=str)
    CONFIGS[name] = res
    return res


def config_scores(name):
    """(y_val, s_val, y_test, s_test) of a recorded config, read back from all_models/<name>/scores.npz."""
    z = np.load(ALL_DIR / name / "scores.npz")
    return z["val_y"], z["val_s"], z["test_y"], z["test_s"]

## Shared helpers and handcrafted_v2 (copied verbatim from `improve_01_original.ipynb`)
The v2 sweep below reloads its cached configs; nothing is retrained.

In [ ]:
import cv2, joblib
from joblib import Parallel, delayed
from scipy.stats import binomtest
from skimage.feature import hog, local_binary_pattern
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.exceptions import ConvergenceWarning


def base_scores(model, split):
    """Scores of a baseline model from its predictions CSV, in this notebook's row order."""
    p = pd.read_csv(BASE_OUT / model / f"predictions_{split}.csv").set_index("relpath").loc[splits[split].relpath]
    assert (p.y_true.values == y_split[split]).all()
    return p[[f"score_{c}" for c in CLASSES]].values


def paired(name_a, s_a, name_b, s_b, y, n_boot=1000):
    """McNemar test + bootstrap 95% CI of the macro-F1 difference (a - b) on the same test images."""
    pa, pb = s_a.argmax(1), s_b.argmax(1)
    a_only, b_only = int(((pa == y) & (pb != y)).sum()), int(((pa != y) & (pb == y)).sum())
    p = binomtest(a_only, a_only + b_only, 0.5).pvalue if a_only + b_only else 1.0
    rng = np.random.default_rng(0)
    d = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        d.append(f1_score(y[i], pa[i], average="macro") - f1_score(y[i], pb[i], average="macro"))
    delta = f1_score(y, pa, average="macro") - f1_score(y, pb, average="macro")
    lo, hi = np.percentile(d, [2.5, 97.5])
    return {"improved": name_a, "baseline": name_b, "improved_only_right": a_only, "baseline_only_right": b_only,
            "mcnemar_p": p, "delta_f1": delta, "ci_lo": lo, "ci_hi": hi, "significant": bool(p < 0.05 and (lo > 0 or hi < 0))}


def choose(prefix_or_names):
    """Config with the best val macro-F1 among the recorded ones."""
    names = [n for n in CONFIGS if n.startswith(prefix_or_names)] if isinstance(prefix_or_names, str) else prefix_or_names
    return max(names, key=lambda n: CONFIGS[n]["val"]["macro_f1"])


PAIRS = []   # (improved model, baseline model) pairs tested at the end

In [ ]:
class GroupScale(BaseEstimator, TransformerMixin):
    """Multiply each column by its group weight (after StandardScaler)."""
    def __init__(self, weights=None):
        self.weights = weights

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X if self.weights is None else X * np.asarray(self.weights, dtype=np.float32)


def make_svm2(C, gamma_mult, weights=None, n_cols=None):
    total_var = float(np.sum(np.square(weights))) if weights is not None else n_cols
    return make_pipeline(StandardScaler(), GroupScale(weights),
                         SVC(C=C, gamma=gamma_mult / total_var, kernel="rbf", decision_function_shape="ovr"))


def group_weights(groups, balanced):
    """groups: list of (name, slice) in column order -> per-column weights (1/sqrt(size) per group) or None."""
    if not balanced:
        return None
    return np.concatenate([np.full(s.stop - s.start, 1 / np.sqrt(s.stop - s.start)) for _, s in groups])


def svm_sweep(feats, feat_name, groups, balanced, Cs, gms, tag):
    """Fit an SVM per (C, gamma_mult) on the chosen column groups; every config is recorded like the team's sweep."""
    cols = np.concatenate([np.arange(s.start, s.stop) for _, s in groups])
    Xs = {sp: feats[sp][:, cols] for sp in ("train", "val", "test")}
    sizes, start, local = [], 0, []
    for g, s in groups:                                   # the same groups, re-indexed inside the selected columns
        n = s.stop - s.start; local.append((g, slice(start, start + n))); start += n
    w = group_weights(local, balanced)
    rows = []
    for C in Cs:
        for gm in gms:
            name = f"svm2_{tag}_C{C:g}_g{gm:g}"
            arch = {"features": feat_name, "groups": [g for g, _ in groups], "balanced": balanced, "C": C, "gamma_mult": gm}
            res = config_done(name, arch)
            if res is None:
                t0 = time.time()
                m = make_svm2(C, gm, w, len(cols)).fit(Xs["train"], y_split["train"])
                res = record_config(name, "svm", "sklearn", m, arch,
                                    {sp: (y_split[sp], m.decision_function(Xs[sp])) for sp in ("val", "test")},
                                    train_f1=macro_f1(y_split["train"], m.decision_function(Xs["train"])))
                print(f"  {name}: train {res['train_f1']:.3f} | val {res['val']['macro_f1']:.3f} | {time.time() - t0:.0f}s")
            rows.append({"tag": tag, "C": C, "gamma_mult": gm, "name": name, "train_f1": res["train_f1"], "val_f1": res["val"]["macro_f1"]})
    return pd.DataFrame(rows)


def load_base_features(split):
    z = np.load(BASE_OUT / f"features_handcrafted_{split}.npz", allow_pickle=True)
    assert list(z["relpath"]) == splits[split].relpath.tolist(), "baseline feature cache is for other images"
    return z["X"]


XB = {sp: load_base_features(sp) for sp in ("train", "val", "test")}
assert XB["train"].shape[1] == 512 + 1764 + 28
BASE_GROUPS = {"color": slice(0, 512), "hog": slice(512, 2276), "lbp": slice(2276, 2304)}
print({sp: v.shape for sp, v in XB.items()})

In [ ]:
# A2: handcrafted_v2 features
GABOR = [cv2.getGaborKernel((21, 21), sigma=0.56 * lam, theta=th, lambd=lam, gamma=0.5, psi=0)
         for lam in (4, 8, 16) for th in np.arange(4) * np.pi / 4]


def handcrafted_v2(path):
    rgb = np.asarray(pad_to_square(Image.open(path).convert("RGB")).resize((128, 128), Image.BILINEAR))   # same input as the baseline
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    hs = cv2.calcHist([hsv], [0, 1], None, [18, 8], [0, 180, 0, 256]).ravel()                # hue x saturation, brightness left out
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    ab = cv2.calcHist([lab], [1, 2], None, [16, 16], [0, 256, 0, 256]).ravel()               # a*b* colour, L left out
    color = np.concatenate([hs / (hs.sum() + 1e-8), ab / (ab.sum() + 1e-8)])
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    shape = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm="L2-Hys")   # as the baseline
    texture = []
    for p, r in ((8, 1), (16, 2), (24, 3)):
        codes = local_binary_pattern(gray, p, r, method="uniform")
        for part in (codes, codes[:64, :64], codes[:64, 64:], codes[64:, :64], codes[64:, 64:]):   # whole image + 2x2 grid
            h, _ = np.histogram(part, bins=p + 2, range=(0, p + 2))
            texture.append(h / h.sum())
    g = gray.astype(np.float32) / 255
    gabor = []
    for k in GABOR:
        resp = np.abs(cv2.filter2D(g, cv2.CV_32F, k))
        gabor += [resp.mean(), resp.std()]
    color, texture = np.sqrt(color), np.sqrt(np.concatenate(texture))                          # Hellinger for histograms
    return np.concatenate([color, shape, texture, np.array(gabor)]).astype(np.float32)


V2_GROUPS = {"color": slice(0, 144 + 256), "hog": slice(400, 400 + 1764), "lbp": slice(2164, 2164 + 270), "gabor": slice(2434, 2434 + 24)}


def load_v2(split):
    cache = OUT / f"features_v2_{split}.npz"
    if cache.exists() and not RETRAIN:
        z = np.load(cache, allow_pickle=True)
        if list(z["relpath"]) == splits[split].relpath.tolist():
            return z["X"]
    t0 = time.time()
    Xv = np.stack(Parallel(n_jobs=-1, batch_size=32)(delayed(handcrafted_v2)(p) for p in splits[split].path))
    np.savez(cache, X=Xv, relpath=splits[split].relpath.values)
    print(f"{split}: v2 features {Xv.shape} in {time.time() - t0:.0f}s")
    return Xv


XV2 = {sp: load_v2(sp) for sp in ("train", "val", "test")}
assert XV2["train"].shape[1] == V2_GROUPS["gabor"].stop
a2 = svm_sweep(XV2, "handcrafted_v2", list(V2_GROUPS.items()), True, SVM2_C, SVM2_GAMMA_MULT, "v2")
display(a2.pivot(index="gamma_mult", columns="C", values="val_f1").style.background_gradient(cmap="Blues").format("{:.3f}"))
a2_nohog = svm_sweep(XV2, "handcrafted_v2", [(g, V2_GROUPS[g]) for g in ("color", "lbp", "gabor")], True, SVM2_C, SVM2_GAMMA_MULT, "v2nohog")
display(a2_nohog.pivot(index="gamma_mult", columns="C", values="val_f1").style.background_gradient(cmap="Blues").format("{:.3f}"))

## Side check: wavelet-energy texture added to handcrafted_v2

A 3-level `db2` wavelet decomposition (lecture 5) of the L, a\*, b\* channels of the same 128 px image.
For each sub-band: mean |coefficient| and std (the energy of that scale and orientation), plus mean and std of the approximation.
3 channels × (3 levels × 3 orientations × 2 + 2) = 60 numbers, as one more balanced group.

In [ ]:
import pywt


def wavelet_energy(path):
    rgb = np.asarray(pad_to_square(Image.open(path).convert("RGB")).resize((128, 128), Image.BILINEAR))   # same input as v2
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32) / 255
    out = []
    for ch in range(3):
        coeffs = pywt.wavedec2(lab[..., ch], "db2", level=3)
        out += [coeffs[0].mean(), coeffs[0].std()]
        for bands in coeffs[1:]:
            for c in bands:
                out += [np.abs(c).mean(), c.std()]
    return np.array(out, dtype=np.float32)


def load_cached(name, fn, split):
    cache = OUT / f"features_{name}_{split}.npz"
    if cache.exists() and not RETRAIN:
        z = np.load(cache, allow_pickle=True)
        if list(z["relpath"]) == splits[split].relpath.tolist():
            return z["X"]
    t0 = time.time()
    X = np.stack(Parallel(n_jobs=-1, batch_size=32)(delayed(fn)(p) for p in splits[split].path))
    np.savez(cache, X=X, relpath=splits[split].relpath.values)
    print(f"{split}: {name} features {X.shape} in {time.time() - t0:.0f}s")
    return X


XW = {sp: load_cached("wavelet", wavelet_energy, sp) for sp in ("train", "val", "test")}
XVW = {sp: np.hstack([XV2[sp], XW[sp]]) for sp in XW}
n2 = XV2["train"].shape[1]
VW_GROUPS = {**V2_GROUPS, "wavelet": slice(n2, n2 + XW["train"].shape[1])}
wav = svm_sweep(XVW, "handcrafted_v2+wavelet", list(VW_GROUPS.items()), True, SVM2_C, SVM2_GAMMA_MULT, "v2wav")
display(wav.pivot(index="gamma_mult", columns="C", values="val_f1").style.background_gradient(cmap="Blues").format("{:.3f}"))
wav_nohog = svm_sweep(XVW, "handcrafted_v2+wavelet", [(g, VW_GROUPS[g]) for g in ("color", "lbp", "gabor", "wavelet")],
                      True, SVM2_C, SVM2_GAMMA_MULT, "v2nohogwav")
display(wav_nohog.pivot(index="gamma_mult", columns="C", values="val_f1").style.background_gradient(cmap="Blues").format("{:.3f}"))

## Colour ablation: is the hue × saturation (HSV) half of the colour group needed?

v2's colour group has two halves: a hue × saturation histogram from HSV (144 bins) and a LAB a\*b\* histogram (256 bins).
Both describe chromaticity without brightness, so one may be redundant. Three variants (all with wavelet):
without HS (a\*b\* is the only colour group), without HS and HOG, and without any colour (to measure what colour is worth).

In [ ]:
HS, AB = slice(0, 144), slice(144, 400)
assert AB.stop == V2_GROUPS["color"].stop
nohs = svm_sweep(XVW, "handcrafted_v2+wavelet", [("ab", AB)] + [(g, VW_GROUPS[g]) for g in ("hog", "lbp", "gabor", "wavelet")],
                 True, SVM2_C, SVM2_GAMMA_MULT, "v2nohs_wav")
nohs_nohog = svm_sweep(XVW, "handcrafted_v2+wavelet", [("ab", AB)] + [(g, VW_GROUPS[g]) for g in ("lbp", "gabor", "wavelet")],
                       True, SVM2_C, SVM2_GAMMA_MULT, "v2nohsnohog_wav")
nocolor = svm_sweep(XVW, "handcrafted_v2+wavelet", [(g, VW_GROUPS[g]) for g in ("hog", "lbp", "gabor", "wavelet")],
                    True, SVM2_C, SVM2_GAMMA_MULT, "v2nocolor_wav")
display(pd.DataFrame({t: [d.val_f1.max()] for t, d in [("with HS (v2wav)", wav), ("no HS", nohs), ("no HS, no HOG", nohs_nohog), ("no colour", nocolor)]},
                     index=["best val macro-F1"]))

## Fisher Vectors: dense RootSIFT + local LAB colour

Step 1: local descriptors. `cv2.SIFT_create().compute` on fixed grid keypoints (no detector), so every image gives the same
number of descriptors at the same places. The colour descriptor is read from box-filtered LAB maps at the 4×4 cell centres of each patch.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.svm import LinearSVC


def dense_local(path):
    """Dense RootSIFT (n×128), local LAB colour (n×48) and normalised patch centres (n×2) of one image."""
    rgb = np.asarray(pad_to_square(Image.open(path).convert("RGB")).resize((FV_SIZE, FV_SIZE), Image.BILINEAR))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32) / 255
    kps, col, xy = [], [], []
    for s in FV_SCALES:
        c = s // 4                                              # 4×4 cells per patch
        box = cv2.blur(lab, (c, c))                             # mean LAB over a c×c cell centred at each pixel
        r = np.arange(s // 2, FV_SIZE - s // 2 + 1, FV_STEP)
        X, Y = [a.ravel() for a in np.meshgrid(r, r)]
        kps += [cv2.KeyPoint(float(x), float(y), float(s)) for x, y in zip(X, Y)]
        off = (np.arange(4) - 1.5) * c
        cx = np.clip(np.rint(X[:, None] + off), 0, FV_SIZE - 1).astype(int)
        cy = np.clip(np.rint(Y[:, None] + off), 0, FV_SIZE - 1).astype(int)
        col.append(box[cy[:, :, None], cx[:, None, :]].reshape(len(X), -1))
        xy.append(np.stack([X, Y], 1))
    kps_out, d = cv2.SIFT_create().compute(gray, kps)
    assert len(kps_out) == len(kps), "SIFT dropped grid keypoints"
    d = np.sqrt(d / (d.sum(1, keepdims=True) + 1e-7))           # RootSIFT
    return d.astype(np.float32), np.concatenate(col).astype(np.float32), (np.concatenate(xy) / FV_SIZE - 0.5).astype(np.float32)


d, c, p = dense_local(splits["train"].path.iloc[0])
print("descriptors per image:", d.shape, c.shape, p.shape)

Step 2: PCA + GMM vocabulary, fit on a random sample of **training** descriptors only (val and test never touch it).

In [ ]:
def fit_codebook():
    cache = OUT / "fv_codebook.joblib"
    key = dict(size=FV_SIZE, step=FV_STEP, scales=FV_SCALES, pca=FV_PCA, K=FV_K, n_img=FV_SAMPLE_IMGS, per_img=FV_SAMPLE_PER_IMG,
               seed=SEED, train=hashlib.sha256("".join(splits["train"].relpath).encode()).hexdigest()[:12])
    if cache.exists() and not RETRAIN:
        cb = joblib.load(cache)
        if cb["key"] == key:
            return cb
    t0 = time.time()
    rng = np.random.default_rng(SEED)
    paths = rng.choice(splits["train"].path.values, min(FV_SAMPLE_IMGS, len(splits["train"])), replace=False)

    def sample(path, seed):
        d, c, p = dense_local(path)
        i = np.random.default_rng(seed).choice(len(d), FV_SAMPLE_PER_IMG, replace=False)
        return d[i], c[i], p[i]

    parts = Parallel(n_jobs=-1, batch_size=16)(delayed(sample)(pth, SEED + j) for j, pth in enumerate(paths))
    desc = {"sift": np.concatenate([a for a, _, _ in parts]), "color": np.concatenate([b for _, b, _ in parts])}
    pos = np.concatenate([q for _, _, q in parts])
    cb = {"key": key, "pca": {}, "gmm": {}}
    for t, D in desc.items():
        pca = PCA(FV_PCA[t], random_state=SEED).fit(desc[t])
        Z = np.hstack([pca.transform(desc[t]), pos]).astype(np.float32)
        cb["pca"][t] = pca
        print(f"{t}: {desc[t].shape[0]:,} sample descriptors, PCA keeps {pca.explained_variance_ratio_.sum():.0%} of variance")
        for K in FV_K:
            gmm = GaussianMixture(K, covariance_type="diag", reg_covar=1e-4, max_iter=200, random_state=SEED).fit(Z)
            cb["gmm"][(t, K)] = gmm
            print(f"  GMM K={K}: converged={gmm.converged_} in {gmm.n_iter_} iterations")
    joblib.dump(cb, cache)
    print(f"codebook in {time.time() - t0:.0f}s")
    return cb


CB = fit_codebook()

Step 3: Fisher Vector of each image. With posteriors γ (descriptor → Gaussian), weights w, means μ and variances σ²,
for each Gaussian k: 𝒢μ = Σ γ (x − μ)/σ ÷ (N√w), 𝒢σ = Σ γ ((x − μ)²/σ² − 1) ÷ (N√(2w)). Then sign·√ and L2.

In [ ]:
def fisher_vector(Z, gmm):
    w, mu, var = gmm.weights_, gmm.means_, gmm.covariances_
    Z = Z.astype(np.float64)
    logp = (-0.5 * ((Z ** 2) @ (1 / var).T - 2 * Z @ (mu / var).T + (mu ** 2 / var).sum(1))
            - 0.5 * np.log(2 * np.pi * var).sum(1) + np.log(w))
    g = np.exp(logp - logp.max(1, keepdims=True)); g /= g.sum(1, keepdims=True)          # posteriors, N×K
    n, s0, s1, s2 = len(Z), g.sum(0), g.T @ Z, g.T @ (Z ** 2)
    sd = np.sqrt(var)
    g_mu = (s1 - mu * s0[:, None]) / sd / (n * np.sqrt(w)[:, None])
    g_var = ((s2 - 2 * mu * s1 + mu ** 2 * s0[:, None]) / var - s0[:, None]) / (n * np.sqrt(2 * w)[:, None])
    fv = np.concatenate([g_mu.ravel(), g_var.ravel()])
    fv = np.sign(fv) * np.sqrt(np.abs(fv))                                                 # power normalisation
    return (fv / (np.linalg.norm(fv) + 1e-12)).astype(np.float32)                          # L2


def encode(path):
    d, c, p = dense_local(path)
    out = {}
    for t, D in (("sift", d), ("color", c)):
        Z = np.hstack([CB["pca"][t].transform(D), p])
        for K in FV_K:
            out[f"{t}_K{K}"] = fisher_vector(Z, CB["gmm"][(t, K)])
    return out


def load_fv(split):
    cache = OUT / f"fv_{split}.npz"
    if cache.exists() and not RETRAIN:
        z = np.load(cache, allow_pickle=True)
        if list(z["relpath"]) == splits[split].relpath.tolist() and str(z["key"]) == str(CB["key"]):
            return {k: z[k] for k in z.files if k not in ("relpath", "key")}
    t0 = time.time()
    rows = Parallel(n_jobs=-1, batch_size=16)(delayed(encode)(p) for p in splits[split].path)
    F = {k: np.stack([r[k] for r in rows]) for k in rows[0]}
    np.savez(cache, relpath=splits[split].relpath.values, key=str(CB["key"]), **F)
    print(f"{split}: Fisher Vectors {({k: v.shape[1] for k, v in F.items()})} in {time.time() - t0:.0f}s")
    return F


FV = {sp: load_fv(sp) for sp in ("train", "val", "test")}

Step 4: linear SVM on the Fisher Vectors. Each FV is already L2-normalised, so concatenating SIFT and colour gives the two equal weight.

In [ ]:
def fv_sweep(parts, K, Cs):
    Xs = {sp: np.hstack([FV[sp][f"{t}_K{K}"] for t in parts]) for sp in FV}
    rows = []
    for C in Cs:
        name = f"svm3_fv_{'+'.join(parts)}_K{K}_C{C:g}"
        arch = {"features": "fisher_vector", "parts": list(parts), "K": K, "C": C, "pca": FV_PCA, "size": FV_SIZE,
                "step": FV_STEP, "scales": list(FV_SCALES), "classifier": "LinearSVC"}
        res = config_done(name, arch)
        if res is None:
            t0 = time.time()
            m = LinearSVC(C=C, max_iter=20000, random_state=SEED).fit(Xs["train"], y_split["train"])
            res = record_config(name, "svm", "sklearn", m, arch,
                                {sp: (y_split[sp], m.decision_function(Xs[sp])) for sp in ("val", "test")},
                                train_f1=macro_f1(y_split["train"], m.decision_function(Xs["train"])))
            print(f"  {name} ({Xs['train'].shape[1]:,} dims): train {res['train_f1']:.3f} | val {res['val']['macro_f1']:.3f} | {time.time() - t0:.0f}s")
        rows.append({"parts": "+".join(parts), "K": K, "C": C, "name": name, "train_f1": res["train_f1"], "val_f1": res["val"]["macro_f1"]})
    return pd.DataFrame(rows)


with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    fvres = pd.concat([fv_sweep(parts, K, FV_C) for K in FV_K for parts in (("sift",), ("color",), ("sift", "color"))], ignore_index=True)
display(fvres.pivot_table(index=["parts", "K"], columns="C", values="val_f1").style.background_gradient(cmap="Blues", axis=None).format("{:.3f}"))

## Chosen model and paired tests

In [ ]:
new = [n for n in CONFIGS if n.startswith(("svm3_", "svm2_v2wav", "svm2_v2nohogwav", "svm2_v2nohs", "svm2_v2nocolor"))]
best = choose(new)
prev = json.loads((OUT / "svm_improved" / "metrics.json").read_text())["config"]
prev_val = json.loads((OUT / "svm_improved" / "metrics.json").read_text())["val"]["macro_f1"]
print(f"chosen: {best}  val {CONFIGS[best]['val']['macro_f1']:.4f}   (svm_improved: val {prev_val:.4f})")
CHOSEN["svm_v3"] = best
report_model("svm_v3", *config_scores(best), CONFIGS[best]["arch"], is_prob=False)


def improved_scores(model, split):
    p = pd.read_csv(OUT / model / f"predictions_{split}.csv").set_index("relpath").loc[splits[split].relpath]
    assert (p.y_true.values == y_split[split]).all()
    return p[[f"score_{c}" for c in CLASSES]].values


s_new = config_scores(best)[3]
pairs = pd.DataFrame([paired("svm_v3", s_new, "svm_improved", improved_scores("svm_improved", "test"), y_split["test"]),
                      paired("svm_v3", s_new, "svm", base_scores("svm", "test"), y_split["test"])])
pairs.to_csv(OUT / "paired_svm_v3.csv", index=False)
display(pairs)

In [ ]:
# every config of this notebook, val (used to choose) and test (for the report)
rows = [{"config": n, "family": "svm", "train_f1": CONFIGS[n]["train_f1"], "val_macro_f1": CONFIGS[n]["val"]["macro_f1"],
         "test_macro_f1": CONFIGS[n]["test"]["macro_f1"]} for n in new]
allc = pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False)
allc.to_csv(OUT / "all_configs_svm_v3.csv", index=False)
display(allc.head(15))

# best val per feature variant, next to the earlier SVM steps
def best_of(prefix):
    """Best-val config with this prefix, read from all_models/ (includes the earlier run's A0/A1 configs)."""
    ms = [json.loads((d / "metrics.json").read_text()) for d in ALL_DIR.iterdir() if d.name.startswith(prefix) and (d / "metrics.json").exists()]
    m = max(ms, key=lambda r: r["val"]["macro_f1"]); return m["model"], m["val"]["macro_f1"], m["test"]["macro_f1"]

variants = [("baseline HSV+HOG+LBP", "svm2_color+hog+lbp_"), ("balanced", "svm2_balanced_"), ("v2", "svm2_v2_"),
            ("v2 + wavelet", "svm2_v2wav_"), ("v2 no HOG + wavelet", "svm2_v2nohogwav_"),
            ("v2 no HS + wavelet", "svm2_v2nohs_wav_"), ("v2 no HS, no HOG + wavelet", "svm2_v2nohsnohog_wav_"), ("v2 no colour + wavelet", "svm2_v2nocolor_wav_")]
variants += [(f"FV {p} K{K}", f"svm3_fv_{p}_K{K}_") for K in FV_K for p in ("sift", "color", "sift+color")]
var = pd.DataFrame([(lab, *best_of(pre)) for lab, pre in variants], columns=["variant", "config", "val_f1", "test_f1"])
var.to_csv(OUT / "svm_v3_variants.csv", index=False)
display(var)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(var.variant[::-1], var.val_f1[::-1], color=["C2" if v.startswith("FV") else "C0" for v in var.variant[::-1]])
for i, v in enumerate(var.val_f1[::-1]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8)
ax.set_xlabel("val macro-F1 (best C per variant)"); ax.set_xlim(0, 1)
ax.set_title("Classic SVM: feature variants on 01_original")
plt.tight_layout(); savefig(fig, "svm_v3_variants"); plt.show()